# 6.1. Clean rooms data
This notebook creates a clean dataset of rooms mapped to labgroupids.

Input:
- Rooms cleaning worksheet
- Individual processed data

Output:
- Cleaned dataset of labgroupids and rooms

In [1]:
# Set-up
import pandas as pd
import numpy as np
import re
import sys
from pathlib import Path
CODE_ROOT = Path.cwd().parents[0]
sys.path.append(str(CODE_ROOT))
import config
from openpyxl import Workbook
from openpyxl.styles import Font, Alignment
import os

# Typos from config file (confidential)
incorrect_value = config.incorrect_value
correct_value = config.correct_value

In [2]:
# Load data
labs = pd.read_csv(
    config.PROCESSED_DATA / "individual_processed_1.csv",
    keep_default_na=False,  # Keep "None" as a string, not NaN
    na_values=[""] # Only treat empty strings as NaN
)

rooms = pd.read_excel(
    config.CLEANING_WORKBOOKS / "rooms_cleaning_workbook_final.xlsx",
    sheet_name="rooms",
    keep_default_na=False,  # Keep "None" as a string, not NaN
    na_values=[""] # Only treat empty strings as NaN
)

## (1) Prepare datasets for merging

In [3]:
# Keep only relevant columns of labs dataframe - labgroupid and all room columns "rooms_1", "rooms_2", etc. plus "rooms_1_co", "rooms_2_co", etc.
cols_to_keep = ["labgroupid"] + [col for col in labs.columns if col.startswith("rooms_")]
labs = labs[cols_to_keep]

# Number of room entries (rooms_1 .. rooms_N)
n_entries = max(int(col.split("_")[1]) for col in labs.columns if col.startswith("rooms_") and not col.endswith("_co"))

# Make labs dataframe long format with columns "labgroupid", "entry_number" (1, 2, etc.), "raw_value" (rooms_i), and "comment" (rooms_i_co)
labs_long = pd.concat([
    labs[["labgroupid", f"rooms_{i}", f"rooms_{i}_co"]]
    .rename(columns={f"rooms_{i}": "raw_value", f"rooms_{i}_co": "comment"})
    .assign(entry_number=i)
    for i in range(1, n_entries + 1)
], ignore_index=True)

# Drop entries with no room reported
labs_long = labs_long.dropna(subset=["raw_value", "comment"], how="all").reset_index(drop=True)

In [4]:
# Keep only relevant columns of rooms dataframe
cols_to_keep_rooms = ["raw_value", "comment", "cleaned_value", "status"]
rooms = rooms[cols_to_keep_rooms]

In [5]:
# Normalize a known typo
def normalize_typo(value):
    return re.sub(rf"^{re.escape(incorrect_value)}-", f"{correct_value}-", value) if isinstance(value, str) else value

labs_long["raw_value"] = labs_long["raw_value"].apply(normalize_typo)
rooms["raw_value"] = rooms["raw_value"].apply(normalize_typo)

## (2) Merge datasets to get cleaned rooms

In [6]:
# For each room entry, merge on raw_value + comment to get the cleaned room value
labs_long = labs_long.merge(rooms, on=["raw_value", "comment"], how="left", validate="m:1", indicator=True)

# Flag entries not yet in the cleaning workbook, so they can be added and cleaned there
unmatched = labs_long[labs_long["_merge"] == "left_only"]
if not unmatched.empty:
    print(f"{len(unmatched)} room entries not found in the cleaning workbook:")
    print(unmatched[["labgroupid", "entry_number", "raw_value", "comment"]].to_string(index=False))

labs_long = labs_long.drop(columns="_merge")

In [7]:
# For all entries with several rooms reported (separated by comma), split into separate rows
labs_long = labs_long.assign(room=labs_long["cleaned_value"].str.split(",")).explode("room").reset_index(drop=True)

In [8]:
# Drop all rows where "room" is missing
labs_long = labs_long.dropna(subset=["room"]).reset_index(drop=True)

## (3) Save cleaned dataset

In [9]:
# Keep relevant columns and save cleaned dataset (one row per labgroupid x room entry)
rooms_clean = labs_long[["labgroupid", "room"]]
rooms_clean.to_csv(config.CLEAN_DATA / "rooms_cleaned.csv", index=False)